# DMPBridge Text-Only Llama Narrative Structure Extraction Test

## Purpose

This notebook tests how well Llama can detect DMP narrative blocks using **text only**.

This version intentionally avoids visual/layout features such as:

- font size
- bold
- color
- x/y position
- indentation
- vertical spacing
- visual hints

The goal is to evaluate Llama's ability to detect:

- document title
- section
- subsection
- content

based only on text, text order, and simple text hierarchy clues.

---

## Workflow

```text
PDFPlumber Extracted JSON Blocks
    ↓
Text Cleaning
    ↓
Text-Only Block Representation
    ↓
Chunk Construction with Context Overlap
    ↓
Llama 3.1 8B Text-Only Labeling
    ↓
Label Validation
    ↓
Post-Processing
    ↓
Structured Narrative JSON
```

---

## Expected Input Folder

```text
data/pdfplumber_extracted_blocks
```

---

## Output Folder

```text
data/llama_structured_blocks_text_only
```

This clean version only saves the final structured block JSON files.

It does **not** save diagnostics files.


## Step 1 — Project root setup

This cell finds your DMPBridge project root and adds `src` to the Python path.


In [1]:
from pathlib import Path
import json
import importlib
import sys
import traceback

# Project root setup first
cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
else:
    project_root = cwd.parent

src_path = str(project_root / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Project root:", project_root)
print("Source path:", src_path)

Project root: c:\Users\Nahid\dmpbridge
Source path: c:\Users\Nahid\dmpbridge\src


In [2]:
from dmpbridge.llm.llama_client import load_llama

# Text-only narrative block module
# Put the updated file here:
# src/dmpbridge/llm/llm_narrative_blocks_text_only.py
import dmpbridge.llm.llm_narrative_blocks_text_only as lnb

from dmpbridge.processing.text_cleaner import clean_repeated_words

# Reload module to avoid old cached notebook version
importlib.reload(lnb)

generate_structured_blocks_with_llm = lnb.generate_structured_blocks_with_llm
save_blocks = lnb.save_blocks

print("Text-only LLM narrative blocks module loaded.")


Text-only LLM narrative blocks module loaded.


In [3]:
pdfplumber_blocks_dir = (
    project_root
    / "data"
    / "pdfplumber_extracted_blocks"
)

# Separate output folder for text-only testing
llama_output_dir = (
    project_root
    / "data"
    / "llama_structured_blocks_text_only"
)

llama_output_dir.mkdir(parents=True, exist_ok=True)

print("PDFPlumber blocks directory:", pdfplumber_blocks_dir)
print("Text-only Llama output directory:", llama_output_dir)


PDFPlumber blocks directory: c:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_blocks
Text-only Llama output directory: c:\Users\Nahid\dmpbridge\data\llama_structured_blocks_text_only


In [4]:
llm = load_llama(
    model_name="llama3.1:8b",
    temperature=0,
)

print("Llama loaded successfully.")


Llama loaded successfully.


## Step 6 — Process all PDFPlumber block JSON files

This cell runs the text-only Llama block detector on every JSON file in:

```text
data/pdfplumber_extracted_blocks
```

For each input file, it saves only the final structured block output:

```text
data/llama_structured_blocks_text_only/sample*_llama_blocks.json
```

No diagnostics files are saved in this clean version.


In [5]:
def get_clean_sample_name(path):
    """
    Create clean sample names.

    Examples:
    sample1.json -> sample1
    sample1_pdfplumber_blocks.json -> sample1
    sample1_extracted_blocks.json -> sample1
    """
    stem = path.stem

    suffixes = [
        "_pdfplumber_blocks",
        "_extracted_blocks",
        "_blocks",
    ]

    for suffix in suffixes:
        if stem.endswith(suffix):
            stem = stem[:-len(suffix)]
            break

    return stem


pdfplumber_block_files = sorted(
    pdfplumber_blocks_dir.glob("*.json")
)

print(f"\nFound {len(pdfplumber_block_files)} PDFPlumber block files")


for block_path in pdfplumber_block_files:
    sample_name = get_clean_sample_name(block_path)

    print("\n" + "=" * 80)
    print(f"Processing: {sample_name}")
    print("=" * 80)

    try:
        # Load PDFPlumber blocks
        with open(block_path, "r", encoding="utf-8") as f:
            pdfplumber_blocks = json.load(f)

        print(f"Original PDFPlumber blocks: {len(pdfplumber_blocks)}")

        # Clean repeated words before LLM structure extraction.
        # This does NOT use font, bold, color, x/y position, indentation,
        # vertical gap, or other visual/layout features.
        cleaned_pdfplumber_blocks = []

        for block in pdfplumber_blocks:
            if not isinstance(block, dict):
                continue

            cleaned_block = dict(block)

            cleaned_text = clean_repeated_words(
                str(block.get("text", ""))
            ).strip()

            if not cleaned_text:
                continue

            cleaned_block["text"] = cleaned_text
            cleaned_pdfplumber_blocks.append(cleaned_block)

        print(f"Cleaned PDFPlumber blocks: {len(cleaned_pdfplumber_blocks)}")

        if not cleaned_pdfplumber_blocks:
            print("Skipped because no valid text blocks were found.")
            continue

        # Main text-only LLM structure extraction
        llama_blocks = generate_structured_blocks_with_llm(
            llm=llm,
            pdf_blocks=cleaned_pdfplumber_blocks,
            return_diagnostics=False,
            chunk_size=45,
            overlap=5,
        )

        llama_output_path = (
            llama_output_dir
            / f"{sample_name}_llama_blocks.json"
        )

        save_blocks(
            blocks=llama_blocks,
            output_path=llama_output_path,
        )

        print(
            f"Saved text-only Llama structured blocks: "
            f"{len(llama_blocks)} blocks -> {llama_output_path.name}"
        )

    except Exception as e:
        print(f"ERROR processing {sample_name}")
        print(type(e).__name__, ":", e)
        traceback.print_exc()


print("\nFinished processing all PDFPlumber block files.")



Found 10 PDFPlumber block files

Processing: sample1
Original PDFPlumber blocks: 79
Cleaned PDFPlumber blocks: 79
Saved text-only Llama structured blocks: 24 blocks -> sample1_llama_blocks.json

Processing: sample10
Original PDFPlumber blocks: 68
Cleaned PDFPlumber blocks: 68
Saved text-only Llama structured blocks: 14 blocks -> sample10_llama_blocks.json

Processing: sample2
Original PDFPlumber blocks: 171
Cleaned PDFPlumber blocks: 171
Saved text-only Llama structured blocks: 33 blocks -> sample2_llama_blocks.json

Processing: sample3
Original PDFPlumber blocks: 69
Cleaned PDFPlumber blocks: 69
Saved text-only Llama structured blocks: 13 blocks -> sample3_llama_blocks.json

Processing: sample4
Original PDFPlumber blocks: 78
Cleaned PDFPlumber blocks: 78
Saved text-only Llama structured blocks: 12 blocks -> sample4_llama_blocks.json

Processing: sample5
Original PDFPlumber blocks: 81
Cleaned PDFPlumber blocks: 81
Saved text-only Llama structured blocks: 33 blocks -> sample5_llama_blo

## Step 7 — Quick output check

This cell lists the structured block files created by the text-only test.


In [6]:
print("\nStructured block outputs:")
for f in sorted(llama_output_dir.glob("*.json")):
    print(f.name)



Structured block outputs:
sample10_llama_blocks.json
sample1_llama_blocks.json
sample2_llama_blocks.json
sample3_llama_blocks.json
sample4_llama_blocks.json
sample5_llama_blocks.json
sample6_llama_blocks.json
sample7_llama_blocks.json
sample8_llama_blocks.json
sample9_llama_blocks.json
